In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
from scipy.stats import norm
import scipy.io

# Load and prepare training set
img_size = (28, 20)
img_data = scipy.io.loadmat('../Data/frey_rawface.mat')["ff"]
img_data = img_data.T.reshape((-1, img_size[0], img_size[1]))
trainX = torch.tensor(img_data[:int(0.8 * img_data.shape[0])], dtype=torch.float)/255.


In [2]:
def get_minibatch(batch_size, device='cpu'):
    indices = torch.randperm(trainX.shape[0])[:batch_size]
    return trainX[indices].reshape(batch_size, -1).to(device)

In [ ]:
class Model(nn.Module):
    def __init__(self, data_dim, context_dim, hidden_dim, constrain_mean):
        self.h=nn.Tanh(nn.Linear(context_dim,hidden_dim))
        self.log_var=nn.Linear(hidden_dim,data_dim)
        self.mu=nn.Linear(hidden_dim,data_dim)
        if constrain_mean:
            self.log_var=nn.Sigmoid(self.mu)
            
    def get_mean_and_log_var(self, x):
        h = self.h(x)
        mu = self.mu(h)
        log_var = self.log_var(h)
        return mu, log_var
